# Qwen3 ASR + TTS API on Google Colab

这个 notebook 复用当前项目根目录里的 `api_server.py` / `asr_engine.py` / `tts_engine.py`，在 Colab GPU 上启动同一套 HTTP API。所有 Colab 相关文件都放在 `colab/` 目录。

In [1]:
USE_GOOGLE_DRIVE = True
DRIVE_REPO_DIR = "/content/drive/MyDrive/qwen-asr-06b"
REPO_DIR = "/content/qwen-asr-06b"
REPO_GIT_URL = ""
REPO_GIT_BRANCH = ""

USE_GOOGLE_DRIVE_FOR_MODELS = True
DRIVE_MODELS_ROOT = "/content/drive/MyDrive/qwen-asr-colab-models"
MODELS_ROOT = "/content/models"
HF_TOKEN = ""

RUNTIME_ROOT = "/content/qwen-colab-runtime"
HOST = "0.0.0.0"
PORT = 8000
TORCH_INSTALL_MODE = "auto"  # auto / force / skip
START_CLOUDFLARED_TUNNEL = False

ENABLE_TTS = True
API_KEY = ""
ASR_LOG_LEVEL = "INFO"
TTS_DEFAULT_SPEAKER = "Vivian"
TTS_DEFAULT_LANGUAGE = "Auto"
TTS_MAX_NEW_TOKENS = 128
TTS_MAX_NEW_TOKENS_LIMIT = 512
TTS_ATTN_IMPLEMENTATION = None

In [2]:
if USE_GOOGLE_DRIVE or USE_GOOGLE_DRIVE_FOR_MODELS:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

Mounted at /content/drive


In [3]:
import os
import shutil
import subprocess
from pathlib import Path

repo_dir = Path(REPO_DIR)
drive_repo_dir = Path(DRIVE_REPO_DIR)

if (repo_dir / "api_server.py").exists():
    print(f"Using existing repo: {repo_dir}")
elif REPO_GIT_URL:
    if repo_dir.exists():
        shutil.rmtree(repo_dir)
    cmd = ["git", "clone", "--depth", "1"]
    if REPO_GIT_BRANCH:
        cmd += ["--branch", REPO_GIT_BRANCH]
    cmd += [REPO_GIT_URL, str(repo_dir)]
    subprocess.check_call(cmd)
    print(f"Cloned repo to: {repo_dir}")
elif DRIVE_REPO_DIR and (drive_repo_dir / "api_server.py").exists():
    if repo_dir.exists():
        shutil.rmtree(repo_dir)
    shutil.copytree(drive_repo_dir, repo_dir)
    print(f"Copied repo from Drive to: {repo_dir}")
else:
    raise RuntimeError(
        "Project source not found. Set REPO_GIT_URL, or put the repo under DRIVE_REPO_DIR, or pre-create REPO_DIR."
    )

os.chdir(repo_dir)
print("cwd=", Path.cwd())

RuntimeError: Project source not found. Set REPO_GIT_URL, or put the repo under DRIVE_REPO_DIR, or pre-create REPO_DIR.

In [ ]:
import sys
sys.path.insert(0, str(repo_dir))

from colab.colab_runtime import install_python_packages, install_system_packages, show_runtime_summary

install_system_packages()
install_python_packages(repo_dir, torch_install_mode=TORCH_INSTALL_MODE)
show_runtime_summary()

In [ ]:
from colab.smoke_test import gpu_summary

gpu_info = gpu_summary()
gpu_info

In [ ]:
from colab.colab_runtime import download_models

models_root = DRIVE_MODELS_ROOT if USE_GOOGLE_DRIVE_FOR_MODELS else MODELS_ROOT
model_info = download_models(
    models_root=models_root,
    include_tts=ENABLE_TTS,
    hf_token=HF_TOKEN or None,
)
model_info

In [ ]:
from colab.colab_runtime import launch_api

service = launch_api(
    repo_dir=repo_dir,
    runtime_root=RUNTIME_ROOT,
    models_root=models_root,
    host=HOST,
    port=PORT,
    enable_tts=ENABLE_TTS,
    api_key=API_KEY,
    asr_log_level=ASR_LOG_LEVEL,
    tts_default_speaker=TTS_DEFAULT_SPEAKER,
    tts_default_language=TTS_DEFAULT_LANGUAGE,
    tts_max_new_tokens=TTS_MAX_NEW_TOKENS,
    tts_max_new_tokens_limit=TTS_MAX_NEW_TOKENS_LIMIT,
    tts_attn_implementation=TTS_ATTN_IMPLEMENTATION,
    start_cloudflared_tunnel_flag=START_CLOUDFLARED_TUNNEL,
)
service.to_dict()

In [ ]:
from colab.smoke_test import healthz

health = healthz(service.local_url, api_key=API_KEY)
health

In [ ]:
from pathlib import Path
from colab.smoke_test import tts_speakers, tts_synthesize_to_file

tts_output = Path(RUNTIME_ROOT) / "artifacts" / "tts_smoke.wav"
if ENABLE_TTS:
    speakers = tts_speakers(service.local_url, api_key=API_KEY)
    tts_result = tts_synthesize_to_file(
        service.local_url,
        output_path=tts_output,
        text="你好，欢迎使用 Qwen3-TTS。",
        speaker=TTS_DEFAULT_SPEAKER,
        language="zh" if TTS_DEFAULT_LANGUAGE == "Auto" else TTS_DEFAULT_LANGUAGE,
        max_new_tokens=TTS_MAX_NEW_TOKENS,
        api_key=API_KEY,
    )
    display({"speakers": speakers, "tts_result": tts_result})
else:
    print("TTS disabled; skipping TTS smoke test.")

In [ ]:
from colab.smoke_test import asr_transcribe_file

sample_asr_audio = repo_dir / "mnt_data" / "input.wav"
if sample_asr_audio.exists():
    asr_result = asr_transcribe_file(
        service.local_url,
        audio_path=sample_asr_audio,
        language="zh",
        max_new_tokens=256,
        api_key=API_KEY,
    )
    asr_result
else:
    print(f"Sample audio not found, skipping ASR smoke test: {sample_asr_audio}")

In [ ]:
from colab.colab_runtime import read_log_tail, stop_service

print(read_log_tail(service.log_path, lines=60))
# stop_service(service)